# 96-well Automated Protein Extraction (APE)

This notebook owns deck setup and `lh.setup()`. The protocol logic lives in `ape_96w_protocol.py` so you can edit the `.py` file and rerun the import + protocol cells without rebuilding the deck each time.

## Do This Before Running

- Put the `TIP_CAR_480_A00` at rails 25 with three 1000 uL tip racks in positions `[0]`, `[1]`, and `[2]`.
- On rails 13, use the 5-position MFX carrier with 10 mm supported DWP holders.
- Rails 13, pos `[0]`: `Alpaqua_96_magnum_flx` magnetic plate adapter.
- Rails 13, pos `[1]`: `AGenBio_1_troughplate_100000uL_Fl` as `WasteTrough`.
- Rails 13, pos `[2]`: `BioER_96_wellplate_Vb_2200uL` as `ElutionPlate`.
- Rails 13, pos `[3]`: `AGenBio_1_troughplate_100000uL_Fl` as `Wash1`.
- Rails 13, pos `[4]`: `AGenBio_1_troughplate_100000uL_Fl` as `Wash2`.
- On rails 19, use the 5-position MFX carrier with standard DWP holders.
- Rails 19, pos `[0]`: `BioER_96_wellplate_Vb_2200uL` as `BindingPlate`.
- Rails 19, pos `[1]`: `BioER_96_wellplate_Vb_2200uL` as `SourcePlate` with lysate supernatants.
- Rails 19, pos `[2]`: `BioER_96_wellplate_Vb_2200uL` as `FlowThrough`.
- Rails 19, pos `[3]`: `AGenBio_1_troughplate_100000uL_Fl` as `BindingTrough` with binding buffer.
- Rails 19, pos `[4]`: `AGenBio_1_troughplate_100000uL_Fl` as `ElutionTrough` with elution buffer.
- Confirm the `BindingPlate` starts empty.
- Confirm the `ElutionPlate` starts empty.
- Confirm `SourcePlate` contains the 96 lysate samples.
- Confirm you have enough binding buffer, wash buffers, and elution buffer loaded in the troughs.
- Be ready to manually add 50 uL magnetic beads to every well of the `BindingPlate` when prompted.


In [1]:
try:
    %load_ext autoreload
    %autoreload 2
except Exception:
    pass

import importlib

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.alpaqua import Alpaqua_96_magnum_flx
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources import hamilton_96_tiprack_1000uL


In [2]:
# Deck layout for the 96-well APE workflow.
backend = STARBackend()
lh: LiquidHandler = LiquidHandler(backend=backend, deck=STARLetDeck())

deck = STARLetDeck(
  core_grippers="1000uL-at-waste"  # or "1000uL-5mL-on-waste"
) 

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)
tiprack_1000_1 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_1000_2 = hamilton_96_tiprack_1000uL("tips_01")
tiprack_1000_3 = hamilton_96_tiprack_1000uL("tips_02")
tip_car[0] = tiprack_1000_1
tip_car[1] = tiprack_1000_2
tip_car[2] = tiprack_1000_3
tip_racks = [tiprack_1000_1, tiprack_1000_2, tiprack_1000_3]

# Rails 13: mag plate + wash / waste / elution resources on 10 mm supports.
rail13_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_mag_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_waste_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_elution_plate_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash1_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("rail13_wash2_module"),
}
car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
lh.deck.assign_child_resource(car_13, rails=13)

mag_plate = Alpaqua_96_magnum_flx("mag_plate")
waste_trough = AGenBio_1_troughplate_100000uL_Fl("waste_trough")
elution_plate = BioER_96_wellplate_Vb_2200uL("elution_plate")
wash1_trough = AGenBio_1_troughplate_100000uL_Fl("wash1_trough")
wash2_trough = AGenBio_1_troughplate_100000uL_Fl("wash2_trough")

rail13_modules[0].assign_child_resource(mag_plate)
rail13_modules[1].assign_child_resource(waste_trough)
rail13_modules[2].assign_child_resource(elution_plate)
rail13_modules[3].assign_child_resource(wash1_trough)
rail13_modules[4].assign_child_resource(wash2_trough)

# Rails 19: binding plate + source / flowthrough / binding buffer / elution buffer.
rail19_modules = {
    0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
    1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
    2: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_flowthrough_module"),
    3: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_trough_module"),
    4: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_elution_trough_module"),
}
car_19 = MFX_CAR_L5_base("car_19", modules=rail19_modules)
lh.deck.assign_child_resource(car_19, rails=19)

binding_plate = BioER_96_wellplate_Vb_2200uL("binding_plate")
source_plate = BioER_96_wellplate_Vb_2200uL("source_plate")
flowthrough_plate = BioER_96_wellplate_Vb_2200uL("flowthrough_plate")
binding_trough = AGenBio_1_troughplate_100000uL_Fl("binding_trough")
elution_trough = AGenBio_1_troughplate_100000uL_Fl("elution_trough")

rail19_modules[0].assign_child_resource(binding_plate)
rail19_modules[1].assign_child_resource(source_plate)
rail19_modules[2].assign_child_resource(flowthrough_plate)
rail19_modules[3].assign_child_resource(binding_trough)
rail19_modules[4].assign_child_resource(elution_trough)


/tmp/ipykernel_934121/1145598107.py:27: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_13 = MFX_CAR_L5_base("car_13", modules=rail13_modules)
/tmp/ipykernel_934121/1145598107.py:30: DeprecationWarning: Alpaqua_96_magnum_flx is deprecated. Use 'alpaqua_96_plateadapter_magnum_flx' instead.
  mag_plate = Alpaqua_96_magnum_flx("mag_plate")
/tmp/ipykernel_934121/1145598107.py:44: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  0: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_binding_module"),
/tmp/ipykernel_934121/1145598107.py:45: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  1: Hamilton_MFX_plateholder_DWP_metal_tapped("rail19_source_module"),
/tmp/ipykernel_934121/1145598107.py:46: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated

In [ ]:
from pylabrobot.liquid_handling.backends.hamilton.STAR_backend import (
    PipChannelInformation,
    STARFirmwareError,
    UnknownHamiltonError,
)

if not getattr(STARBackend, "_ape_legacy_vw_compat", False):
    _original_pip_channel_request_configuration = STARBackend._pip_channel_request_configuration

    async def _compat_pip_channel_request_configuration(self, channel):
        try:
            return await _original_pip_channel_request_configuration(self, channel)
        except STARFirmwareError as exc:
            if all(
                isinstance(err, UnknownHamiltonError) and err.message == "Unknown command"
                for err in exc.errors.values()
            ):
                return PipChannelInformation(
                    channel_type="ML_STAR",
                    head_type="ML_STAR",
                    stop_disc_type="core_i",
                    pressure_adc="Renesas_X9268",
                )
            raise

    STARBackend._pip_channel_request_configuration = _compat_pip_channel_request_configuration
    STARBackend._ape_legacy_vw_compat = True

await lh.setup(skip_autoload=True)

# STARlet without iSWAP reports 0 arms; keep Co-Re gripper bookkeeping available.
if lh.backend.num_arms == 0:
    lh._resource_pickups = {0: None}

print(lh.deck.get_resource("core_grippers"))


In [4]:
# PLR deck setup overview.
print(lh.summary())


Rail  Resource                      Type                 Coordinates (mm)
(-6)  ├── trash_core96              Trash                (-58.200, 106.000, 216.400)
      │
(13)  ├── car_13                    MFXCarrier           (370.000, 063.000, 100.000)
      │   ├── mag_plate             PlateAdapter         (374.000, 072.000, 138.195)
      │   ├── waste_trough          Plate                (374.000, 168.000, 133.455)
      │   ├── elution_plate         Plate                (374.000, 264.000, 133.455)
      │   ├── wash1_trough          Plate                (374.000, 360.000, 133.455)
      │   ├── wash2_trough          Plate                (374.000, 456.000, 133.455)
      │
(19)  ├── car_19                    MFXCarrier           (505.000, 063.000, 100.000)
      │   ├── binding_plate         Plate                (509.000, 072.000, 179.210)
      │   ├── source_plate          Plate                (509.000, 168.000, 179.210)
      │   ├── flowthrough_plate     Plate                (50

In [5]:
import ape_96w_protocol

ape_96w_protocol = importlib.reload(ape_96w_protocol)


In [ ]:
import ape_96w_protocol

ape_96w_protocol = importlib.reload(ape_96w_protocol)

await ape_96w_protocol.run_ape_96w(
    lh,
    binding_plate=binding_plate,
    source_plate=source_plate,
    flowthrough_plate=flowthrough_plate,
    elution_plate=elution_plate,
    mag_plate=mag_plate,
    binding_trough=binding_trough,
    wash1_trough=wash1_trough,
    wash2_trough=wash2_trough,
    waste_trough=waste_trough,
    elution_trough=elution_trough,
    tip_racks=tip_racks,
    lysate_volume=300,
    binding_buffer_volume=1500,
    bead_volume_note=50,
    wash_volume=900,
    elution_volume=120,
    binding_mix_rounds=6,
    binding_incubation_seconds=30,
    wash_mix_rounds=1,
    wash1_cycles=1,
    magnet_settle_seconds=20,
    wash_incubation_seconds=2,
    elution_incubation_seconds=120,
    fill_binding_plate=True,
    pause_for_beads=True,
    transfer_lysate=True,
    mix_binding_step=True,
    remove_binding_supernatant=True,
    run_wash1=True,
    run_wash2=True,
    run_final_wash=True,
    add_elution_buffer=True,
    mix_elution_step=True,
    recover_elution=True,
)


Starting APE 96-well protocol.
Skipping binding plate fill; assuming BindingPlate is already pre-filled.
Skipping lysate transfer; assuming BindingPlate already contains lysate.
Skipping lysate/bead mixing.
Skipping magnet capture and binding supernatant removal.
Skipping Wash 1.
Wash 2 added cycle 1: column 1/12 (900 uL).
Wash 2 added cycle 1: column 2/12 (900 uL).
Wash 2 added cycle 1: column 3/12 (900 uL).
Wash 2 added cycle 1: column 4/12 (900 uL).
Wash 2 added cycle 1: column 5/12 (900 uL).
Wash 2 added cycle 1: column 6/12 (900 uL).
Wash 2 added cycle 1: column 7/12 (900 uL).
Wash 2 added cycle 1: column 8/12 (900 uL).
Wash 2 added cycle 1: column 9/12 (900 uL).
Wash 2 added cycle 1: column 10/12 (900 uL).
Wash 2 added cycle 1: column 11/12 (900 uL).
Wash 2 added cycle 1: column 12/12 (900 uL).
Wash 2 mix cycle 1: round 1/1
Using Co-Re gripper channels 3 and 4.
Cleared wash from column 1/12.
Cleared wash from column 2/12.
Cleared wash from column 3/12.
Cleared wash from column 4/

In [ ]:
lh.summary()

In [ ]:
# await lh.drop_tips(tiprack_1000_1["A12:F12"], use_channels=[0,1,2,3,4,5])
# await lh.drop_tips(tiprack_1000_2["A1:H1"], use_channels=[0,1,2,3,4,5,6,7])
# await lh.dispense(
#                 elution_trough["A1"]*8,
#                 vols=[120]*8,
#                 use_channels=[0,1,2,3,4,5,6,7],
#                 liquid_height = [8]*8, # the perfect height for 1000ul. About 1mm when finished. 
#                 flow_rates=[200]*8,
#                 # auto_surface_following_distance=True,
#                 blow_out=[1]*8, 
#                 swap_speed=[160]*8,
#                 settling_time=[1]*8
#             )
# await lh.drop_tips(tiprack_1000_3["A1:H1"], use_channels=[0,1,2,3,4,5,6,7])
# await lh.backend.return_core_gripper_tools()


In [ ]:
# # move from mag plate to rail10[0]
# await lh.move_plate(
#     plate=binding_plate,
#     to=rail19_modules[0],   # or whatever empty holder is actually empty
#     use_arm="core",
#     pickup_distance_from_top=10,
#     channel_1=3,
#     channel_2=4,
#     core_grip_strength=60,
#     enable_recovery=False,
#     return_core_gripper=False,
# )
# await lh.backend.return_core_gripper_tools()

# await lh.move_plate(
#     plate=binding_plate,
#     to=mag_plate,   # or whatever empty holder is actually empty
#     use_arm="core",
#     pickup_distance_from_top=10,
#     channel_1=3,
#     channel_2=4,
#     core_grip_strength=60,
#     enable_recovery=False,
#     return_core_gripper=False,
# )
# await lh.backend.return_core_gripper_tools()
# lh.stop()
# 

<coroutine object Machine.stop at 0x75c0bf304f40>